# simple_agents_framework on Colab

Markdown file in, callable agent out. Run the cells top to bottom.

Repo: https://github.com/Agentsable/simple_agents_framework

## 1. Install

The `claude` CLI the SDK shells out to is bundled in the pip package, so this is
the only install step.

In [ ]:
%pip install -q git+https://github.com/Agentsable/simple_agents_framework.git

## 2. API key

Paste it below, or leave it blank to use `ANTHROPIC_API_KEY` from the environment or be prompted (keeps it out of the notebook file).

In [ ]:
import getpass
import os

API_KEY = ''.strip() or os.environ.get('ANTHROPIC_API_KEY') or getpass.getpass('Anthropic API key: ')
assert API_KEY.startswith('sk-'), 'that does not look like an API key'

## 3. Define an agent

Frontmatter configures it, the body is the system prompt. `permission_mode: acceptEdits`
is needed on Colab: Colab runs as root, and the CLI refuses `bypassPermissions` there.

In [ ]:
from pathlib import Path

Path('doc_reader.md').write_text('''---
name: doc_reader
model: claude-sonnet-5-5
tools: Read, Grep, Glob
permission_mode: acceptEdits
---
You answer questions about the files in this folder.

- Glob/Grep to find the relevant files and Read them before answering. Never guess.
- At most 5 bullets, one sentence each, citing locations as `file:line`.
''')

## 4. Ask it something

`stream=html_stream` renders the run live: blue sent, green agent, amber tool call,
cyan tool result. Drop `stream=` to just get the string back.

In [ ]:
import simple_agents_framework as saf
from plugins.streaming.jupyter_html import html_stream

WORKDIR = '/content' if os.path.isdir('/content') else '.'  # Colab, or wherever you run it
agent = saf.create_agent_from_markdown('doc_reader.md', API_KEY, cwd=WORKDIR)
agent.ask('What files are in this folder, and what is each one for?', stream=html_stream)

Follow-ups resume the same session, so context carries over.

In [ ]:
agent.ask('Which of those changed most recently?', stream=html_stream)